# Comparación de predicciones por día de la semana

Comparamos las configuraciones ya probadas en los notebooks de baseline, SARIMAX, Prophet y LightGBM. Todas predicen 2025 de una sola vez con información anterior al año evaluado; las métricas se calculan sobre los mismos días que realmente abrieron.

Las predicciones individuales no están todas guardadas en los notebooks anteriores. Por eso esta comparación **vuelve a ejecutar sus funciones y ajustes**, con las mismas configuraciones, sin añadir modelos nuevos. Algunos ajustes SARIMAX pueden tardar varios minutos. Los datos se leen del Gold local y las salidas muestran solo índices, porcentajes y recuentos.

Mostramos MAPE, WAPE, MAE y sesgo por día de la semana, por grupos de días y por mes. Las clasificaciones son **descriptivas de 2025**: elegir un modelo mirando estos resultados no constituye una evaluación independiente de esa elección. Especialmente, solo hay cinco domingos abiertos en 2025.

In [1]:
import ast
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
NOTEBOOKS = PROJECT_ROOT / "notebooks"
INICIO, FIN = "2025-01-01", "2025-12-31"
DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

def celda_por_inicio(notebook, prefijo):
    coincidencias = [celda.source for celda in notebook.cells if celda.cell_type == "code" and celda.source.startswith(prefijo)]
    if len(coincidencias) != 1:
        raise ValueError(f"Se esperaba una única celda que comienza por {prefijo!r}; hay {len(coincidencias)}.")
    return coincidencias[0]

def cargar_funciones(nombre, prefijos):
    ruta = NOTEBOOKS / nombre
    notebook = nbformat.read(ruta, as_version=4)
    espacio = {"__name__": "comparacion_notebook"}
    for prefijo in prefijos:
        codigo = celda_por_inicio(notebook, prefijo)
        exec(compile(codigo, str(ruta), "exec"), espacio)
    return notebook, espacio

def cargar_configuraciones_sarimax(notebook, espacio):
    codigo = celda_por_inicio(notebook, "CONFIGURACIONES = {")
    asignaciones = [nodo for nodo in ast.parse(codigo).body if isinstance(nodo, ast.Assign) and any(isinstance(destino, ast.Name) and destino.id == "CONFIGURACIONES" for destino in nodo.targets)]
    if len(asignaciones) != 1:
        raise ValueError("No se encontró una asignación única de configuraciones SARIMAX.")
    exec(compile(ast.Module(body=asignaciones, type_ignores=[]), "configuraciones_sarimax", "exec"), espacio)

resultados = []
def agregar_resultado(familia, configuracion, entrenamiento, resultado, columna):
    columnas = resultado[["fecha", "indice", columna]].rename(columns={columna: "prediccion"}).copy()
    columnas["familia"] = familia
    columnas["configuracion"] = configuracion
    columnas["entrenamiento"] = entrenamiento
    resultados.append(columnas)

In [2]:
print("Reproduciendo baselines...")
_, baseline = cargar_funciones("02_pruebas_baseline.ipynb", ["from pathlib import Path", "ventas = ventas.loc[", "DIAS = ", "def fecha_equivalente("])
comparacion_baselines = baseline["evaluar_baselines"](baseline["abiertos"], INICIO, FIN)
for nombre, resultado in comparacion_baselines.groupby("modelo", sort=False):
    agregar_resultado("Baseline", nombre, "2022–2024", resultado, "prediccion")

print("Reproduciendo configuraciones SARIMAX; puede tardar varios minutos...")
notebook_sarimax, sarimax = cargar_funciones("03_pruebas_SARIMAX.ipynb", ["from pathlib import Path", "DIAS = "])
cargar_configuraciones_sarimax(notebook_sarimax, sarimax)
for nombre, (orden, orden_estacional) in sarimax["CONFIGURACIONES"].items():
    resultado, _ = sarimax["evaluar_periodo"](sarimax["datos"], INICIO, FIN, orden, orden_estacional)
    agregar_resultado("SARIMAX", nombre, "2022–2024", resultado, "sarimax")
resultado, _ = sarimax["evaluar_periodo"](sarimax["datos"], INICIO, FIN, interaccion_sabado=True)
agregar_resultado("SARIMAX", "Inicial + sábado en temporada alta", "2022–2024", resultado, "sarimax")
for desde, etiqueta in [("2023-01-01", "2023–2024"), ("2024-01-01", "2024")]:
    ventana = sarimax["datos"].loc[desde:FIN]
    resultado, _ = sarimax["evaluar_periodo"](ventana, INICIO, FIN)
    agregar_resultado("SARIMAX", "Inicial", etiqueta, resultado, "sarimax")

print("Reproduciendo configuraciones Prophet...")
_, prophet = cargar_funciones("04_pruebas_Prophet.ipynb", ["from pathlib import Path", "DIAS = "])
for nombre, configuracion in prophet["CONFIGURACIONES"].items():
    resultado, _ = prophet["evaluar_periodo"](prophet["datos"], INICIO, FIN, configuracion)
    agregar_resultado("Prophet", nombre, "2022–2024", resultado, "prophet")
nombre_prophet = "Plana con días como regresores"
for desde, etiqueta in [("2023-01-01", "2023–2024"), ("2024-01-01", "2024")]:
    resultado, _ = prophet["evaluar_periodo"](prophet["datos"], INICIO, FIN, prophet["CONFIGURACIONES"][nombre_prophet], desde)
    agregar_resultado("Prophet", nombre_prophet, etiqueta, resultado, "prophet")

print("Reproduciendo configuraciones LightGBM...")
_, lightgbm = cargar_funciones("05_pruebas_LightGBM.ipynb", ["from pathlib import Path", "DIAS = "])
for nombre, configuracion in lightgbm["CONFIGURACIONES"].items():
    resultado, _ = lightgbm["evaluar_periodo"](lightgbm["datos"], INICIO, FIN, configuracion)
    agregar_resultado("LightGBM", nombre, "2022–2024", resultado, "lightgbm")
for desde, etiqueta in [("2023-01-01", "2023–2024"), ("2024-01-01", "2024")]:
    resultado, _ = lightgbm["evaluar_periodo"](lightgbm["datos"], INICIO, FIN, lightgbm["CONFIGURACIONES"]["Inicial"], desde)
    agregar_resultado("LightGBM", "Inicial", etiqueta, resultado, "lightgbm")

predicciones = pd.concat(resultados, ignore_index=True)
identidad = ["familia", "configuracion", "entrenamiento"]
referencia = predicciones.loc[(predicciones["familia"] == "Baseline") & (predicciones["configuracion"] == "Mediana día semanal")].sort_values("fecha")
if referencia.empty or referencia["fecha"].duplicated().any():
    raise ValueError("La referencia no contiene fechas abiertas únicas.")
for clave, grupo in predicciones.groupby(identidad, sort=False):
    grupo = grupo.sort_values("fecha")
    if grupo["fecha"].duplicated().any() or not grupo["fecha"].reset_index(drop=True).equals(referencia["fecha"].reset_index(drop=True)):
        raise ValueError(f"Las fechas evaluadas de {clave} no coinciden con la referencia.")
    if not np.allclose(grupo["indice"], referencia["indice"], rtol=1e-12, atol=1e-12):
        raise ValueError(f"Los índices reales de {clave} no coinciden con la referencia.")
    if not np.isfinite(grupo[["indice", "prediccion"]].to_numpy()).all() or grupo["indice"].le(0).any():
        raise ValueError(f"Hay índices o predicciones no válidos en {clave}.")
predicciones["numero_dia"] = predicciones["fecha"].dt.day_of_week
predicciones["grupo_dia"] = np.where(predicciones["numero_dia"] < 5, "Lunes–viernes", np.where(predicciones["numero_dia"] == 5, "Sábado", "Domingo"))
print(f"Configuraciones comparadas: {len(resultados)}; días abiertos por configuración: {len(referencia)}.")
print("Comprobado: todas se evalúan sobre las mismas fechas y valores reales.")

Reproduciendo baselines...


,Incidencias
Fechas inválidas,0
Fechas duplicadas,0
Fechas fuera del periodo,0
Fechas ausentes,0
Totales no numéricos o infinitos,0
Totales negativos,0


Controles superados.


,Días abiertos
Año,
2022,355
2023,311
2024,307
2025,308


Índice calculado con base en los días abiertos de 2022.
Reproduciendo configuraciones SARIMAX; puede tardar varios minutos...


,Incidencias
Fechas inválidas,0
Fechas duplicadas,0
Fechas fuera del periodo,0
Fechas ausentes,0
Totales no numéricos o infinitos,0
Totales negativos,0
Festivos no válidos,0


... (3 outputs left out)

,Incidencias
Fechas inválidas,0
Fechas duplicadas,0
Fechas fuera del periodo,0
Fechas ausentes,0
Totales no numéricos o infinitos,0
Totales negativos,0
Festivos no válidos,0


Controles superados. Índice calculado; los cierres no se usan como observaciones.
Reproduciendo configuraciones LightGBM...


,Incidencias
Fechas inválidas,0
Fechas duplicadas,0
Fechas fuera del periodo,0
Fechas ausentes,0
Totales no numéricos o infinitos,0
Totales negativos,0
Festivos no válidos,0


Controles superados. Índice calculado; los cierres no se usan como observaciones.
Configuraciones comparadas: 30; días abiertos por configuración: 308.
Comprobado: todas se evalúan sobre las mismas fechas y valores reales.


## Comparación por día

El MAPE promedia el error porcentual diario; el WAPE da más peso a los días con mayor índice real. El sesgo es predicción menos realidad. Primero resumimos el baseline y la configuración de menor MAPE de cada día; después se muestran todas las configuraciones. El WAPE mostrado en el resumen corresponde a esa configuración y no tiene por qué ser el mejor. Un valor ligeramente menor no demuestra una diferencia práctica.

In [6]:
def metricas_grupo(grupo):
    error = grupo["prediccion"] - grupo["indice"]
    return pd.Series({"Días abiertos": len(grupo), "MAE (puntos de índice)": error.abs().mean(),
                      "MAPE (%)": 100 * error.abs().div(grupo["indice"]).mean(),
                      "WAPE (%)": 100 * error.abs().sum() / grupo["indice"].sum(),
                      "Sesgo (puntos de índice)": error.mean()})

filas_dias = []
for clave, grupo in predicciones.groupby(["numero_dia"] + identidad, sort=False):
    numero, familia, configuracion, entrenamiento = clave
    filas_dias.append({"Día": DIAS[numero], "Familia": familia, "Configuración": configuracion,
                       "Entrenamiento": entrenamiento, **metricas_grupo(grupo).to_dict()})
tabla_dias = pd.DataFrame(filas_dias)
tabla_dias["Días abiertos"] = tabla_dias["Días abiertos"].astype(int)

def resumir_mejores(tabla, columna, grupos):
    resumen = []
    for nombre in grupos:
        grupo = tabla.loc[tabla[columna].eq(nombre)]
        if grupo.empty:
            raise ValueError(f"No hay días abiertos para evaluar {nombre}.")
        referencia = grupo.loc[(grupo["Familia"] == "Baseline") & (grupo["Configuración"] == "Mediana día semanal")]
        if len(referencia) != 1:
            raise ValueError(f"Falta una referencia única para {nombre}.")
        base = referencia.iloc[0]
        mejor = grupo.loc[grupo["MAPE (%)"].idxmin()]
        resumen.append({columna: nombre, "Días abiertos": int(base["Días abiertos"]),
                        "Baseline MAPE (%)": base["MAPE (%)"], "Baseline WAPE (%)": base["WAPE (%)"],
                        "Mejor familia por MAPE": mejor["Familia"], "Mejor configuración por MAPE": mejor["Configuración"],
                        "Entrenamiento": mejor["Entrenamiento"], "Mejor MAPE (%)": mejor["MAPE (%)"],
                        "WAPE de esa configuración (%)": mejor["WAPE (%)"]})
    return pd.DataFrame(resumen).set_index(columna)

print("Resumen por día de la semana:")
display(resumir_mejores(tabla_dias, "Día", DIAS).round(2))

for dia in DIAS:
    print(f"\n{dia}:")
    tabla = tabla_dias.loc[tabla_dias["Día"].eq(dia)].sort_values("MAPE (%)", kind="stable")
    with pd.option_context("display.max_rows", None):
        display(tabla.drop(columns="Día").set_index(["Familia", "Configuración", "Entrenamiento"]).round(2))

Resumen por día de la semana:


,Días abiertos,Baseline MAPE (%),Baseline WAPE (%),Mejor familia por MAPE,Mejor configuración por MAPE,Entrenamiento,Mejor MAPE (%),WAPE de esa configuración (%)
Día,,,,,,,,
Lunes,50,36.41,29.32,LightGBM,Error absoluto,2022–2024,33.28,27.14
Martes,51,35.15,29.14,Baseline,Mediana día semanal,2022–2024,35.15,29.14
Miércoles,52,33.75,29.46,LightGBM,Inicial,2024,33.14,29.06
Jueves,49,30.41,27.82,Baseline,Mediana día semanal,2022–2024,30.41,27.82
Viernes,50,33.11,28.46,Prophet,Plana sin patrón semanal,2022–2024,29.68,26.72
Sábado,51,19.23,18.45,Baseline,Mediana día semanal,2022–2024,19.23,18.45
Domingo,5,79.74,69.87,Prophet,Plana con días como regresores,2024,53.72,61.62



Lunes:


Días abiertos  \
Familia  Configuración                      Entrenamiento                  
LightGBM Error absoluto                     2022–2024                 50   
         Más detalle                        2022–2024                 50   
         Calendario ampliado                2022–2024                 50   
SARIMAX  Inicial + sábado en temporada alta 2022–2024                 50   
LightGBM Más simple                         2022–2024                 50   
SARIMAX  Inicial                            2022–2024                 50   
         AR + SMA semanal                   2022–2024                 50   
Baseline Mediana día semanal                2022–2024                 50   
SARIMAX  AR + SAR semanal                   2022–2024                 50   
         SAR semanal                        2022–2024                 50   
Prophet  Tendencia plana + anual suave      2022–2024                 50   
         Plana con días como regresores     2022–2024                 50   
         Tendencia plana                    2022–2024                 50   
Baseline Mediana anual                      2022–2024                 50   
Prophet  Tendencia plana + sábado           2022–2024                 50   
LightGBM Inicial                            2022–2024                 50   
Baseline Media anual                        2022–2024                 50   
Prophet  Inicial                            2022–2024                 50   
Baseline Mediana mes y día                  2022–2024                 50   
Prophet  Sábado en temporada alta           2022–2024                 50   
         Estacionalidad anual suave         2022–2024                 50   
         Tendencia regularizada             2022–2024                 50   
         Plana con días como regresores     2023–2024                 50   
SARIMAX  Inicial                            2023–2024                 50   
LightGBM Inicial                            2023–2024                 50   
Prophet  Plana con días como regresores     2024                      50   
SARIMAX  Inicial                            2024                      50   
LightGBM Inicial                            2024                      50   
Baseline Naïve anual                        2022–2024                 50   
Prophet  Plana sin patrón semanal           2022–2024                 50   

                                                           MAE (puntos de índice)  \
Familia  Configuración                      Entrenamiento                           
LightGBM Error absoluto                     2022–2024                       23.12   
         Más detalle                        2022–2024                       23.57   
         Calendario ampliado                2022–2024                       24.37   
SARIMAX  Inicial + sábado en temporada alta 2022–2024                       24.31   
LightGBM Más simple                         2022–2024                       23.34   
SARIMAX  Inicial                            2022–2024                       24.34   
         AR + SMA semanal                   2022–2024                       24.26   
Baseline Mediana día semanal                2022–2024                       24.97   
SARIMAX  AR + SAR semanal                   2022–2024                       24.29   
         SAR semanal                        2022–2024                       24.30   
Prophet  Tendencia plana + anual suave      2022–2024                       24.13   
         Plana con días como regresores     2022–2024                       24.25   
         Tendencia plana                    2022–2024                       24.25   
Baseline Mediana anual                      2022–2024                       25.00   
Prophet  Tendencia plana + sábado           2022–2024                       24.38   
LightGBM Inicial                            2022–2024                       24.79   
Baseline Media anual                        2022–2024                       25.71   
Prophet  In


Martes:


Días abiertos  \
Familia  Configuración                      Entrenamiento                  
Baseline Mediana día semanal                2022–2024                 51   
LightGBM Calendario ampliado                2022–2024                 51   
         Más simple                         2022–2024                 51   
Baseline Mediana mes y día                  2022–2024                 51   
LightGBM Error absoluto                     2022–2024                 51   
         Más detalle                        2022–2024                 51   
SARIMAX  Inicial                            2023–2024                 51   
Prophet  Tendencia plana + sábado           2022–2024                 51   
SARIMAX  Inicial                            2022–2024                 51   
Prophet  Plana con días como regresores     2022–2024                 51   
         Tendencia plana                    2022–2024                 51   
SARIMAX  Inicial + sábado en temporada alta 2022–2024                 51   
         SAR semanal                        2022–2024                 51   
         AR + SAR semanal                   2022–2024                 51   
         AR + SMA semanal                   2022–2024                 51   
Prophet  Plana con días como regresores     2023–2024                 51   
LightGBM Inicial                            2023–2024                 51   
Prophet  Plana con días como regresores     2024                      51   
         Sábado en temporada alta           2022–2024                 51   
         Inicial                            2022–2024                 51   
         Tendencia plana + anual suave      2022–2024                 51   
LightGBM Inicial                            2022–2024                 51   
                                            2024                      51   
SARIMAX  Inicial                            2024                      51   
Prophet  Estacionalidad anual suave         2022–2024                 51   
         Tendencia regularizada             2022–2024                 51   
         Plana sin patrón semanal           2022–2024                 51   
Baseline Media anual                        2022–2024                 51   
         Mediana anual                      2022–2024                 51   
         Naïve anual                        2022–2024                 51   

                                                           MAE (puntos de índice)  \
Familia  Configuración                      Entrenamiento                           
Baseline Mediana día semanal                2022–2024                       26.57   
LightGBM Calendario ampliado                2022–2024                       27.29   
         Más simple                         2022–2024                       27.41   
Baseline Mediana mes y día                  2022–2024                       28.60   
LightGBM Error absoluto                     2022–2024                       28.32   
         Más detalle                        2022–2024                       28.86   
SARIMAX  Inicial                            2023–2024                       27.93   
Prophet  Tendencia plana + sábado           2022–2024                       28.34   
SARIMAX  Inicial                            2022–2024                       27.98   
Prophet  Plana con días como regresores     2022–2024                       28.41   
         Tendencia plana                    2022–2024                       28.41   
SARIMAX  Inicial + sábado en temporada alta 2022–2024                       27.94   
         SAR semanal                        2022–2024                       28.02   
         AR + SAR semanal                   2022–2024                       28.04   
         AR + SMA semanal                   2022–2024                       28.06   
Prophet  Plana con días como regresores     2023–2024                       29.05   
LightGBM Inicial                            2023–2024                       28.86   
Prophet  Pl

... (6 outputs left out)


Sábado:


Días abiertos  \
Familia  Configuración                      Entrenamiento                  
Baseline Mediana día semanal                2022–2024                 51   
LightGBM Más simple                         2022–2024                 51   
         Error absoluto                     2022–2024                 51   
         Inicial                            2022–2024                 51   
Baseline Mediana mes y día                  2022–2024                 51   
LightGBM Más detalle                        2022–2024                 51   
Prophet  Tendencia plana + anual suave      2022–2024                 51   
         Plana con días como regresores     2022–2024                 51   
         Tendencia plana                    2022–2024                 51   
         Inicial                            2022–2024                 51   
LightGBM Inicial                            2024                      51   
SARIMAX  SAR semanal                        2022–2024                 51   
         AR + SAR semanal                   2022–2024                 51   
         Inicial                            2022–2024                 51   
LightGBM Calendario ampliado                2022–2024                 51   
SARIMAX  AR + SMA semanal                   2022–2024                 51   
Prophet  Estacionalidad anual suave         2022–2024                 51   
         Sábado en temporada alta           2022–2024                 51   
         Tendencia plana + sábado           2022–2024                 51   
LightGBM Inicial                            2023–2024                 51   
SARIMAX  Inicial + sábado en temporada alta 2022–2024                 51   
Baseline Media anual                        2022–2024                 51   
Prophet  Tendencia regularizada             2022–2024                 51   
Baseline Mediana anual                      2022–2024                 51   
SARIMAX  Inicial                            2024                      51   
Prophet  Plana con días como regresores     2024                      51   
                                            2023–2024                 51   
SARIMAX  Inicial                            2023–2024                 51   
Baseline Naïve anual                        2022–2024                 51   
Prophet  Plana sin patrón semanal           2022–2024                 51   

                                                           MAE (puntos de índice)  \
Familia  Configuración                      Entrenamiento                           
Baseline Mediana día semanal                2022–2024                       29.84   
LightGBM Más simple                         2022–2024                       30.98   
         Error absoluto                     2022–2024                       30.62   
         Inicial                            2022–2024                       30.78   
Baseline Mediana mes y día                  2022–2024                       31.84   
LightGBM Más detalle                        2022–2024                       31.91   
Prophet  Tendencia plana + anual suave      2022–2024                       31.59   
         Plana con días como regresores     2022–2024                       31.60   
         Tendencia plana                    2022–2024                       31.61   
         Inicial                            2022–2024                       31.38   
LightGBM Inicial                            2024                            30.78   
SARIMAX  SAR semanal                        2022–2024                       32.01   
         AR + SAR semanal                   2022–2024                       32.02   
         Inicial                            2022–2024                       32.07   
LightGBM Calendario ampliado                2022–2024                       33.04   
SARIMAX  AR + SMA semanal                   2022–2024                       32.06   
Prophet  Estacionalidad anual suave         2022–2024                       31.69   
         Sá


Domingo:


Días abiertos  \
Familia  Configuración                      Entrenamiento                  
Prophet  Plana con días como regresores     2024                       5   
SARIMAX  Inicial                            2024                       5   
Prophet  Plana con días como regresores     2023–2024                  5   
SARIMAX  Inicial                            2023–2024                  5   
Baseline Mediana día semanal                2022–2024                  5   
         Naïve anual                        2022–2024                  5   
LightGBM Más detalle                        2022–2024                  5   
         Error absoluto                     2022–2024                  5   
Baseline Mediana mes y día                  2022–2024                  5   
LightGBM Calendario ampliado                2022–2024                  5   
         Más simple                         2022–2024                  5   
Prophet  Tendencia plana + anual suave      2022–2024                  5   
SARIMAX  SAR semanal                        2022–2024                  5   
         AR + SAR semanal                   2022–2024                  5   
         AR + SMA semanal                   2022–2024                  5   
         Inicial                            2022–2024                  5   
LightGBM Inicial                            2022–2024                  5   
SARIMAX  Inicial + sábado en temporada alta 2022–2024                  5   
Prophet  Tendencia plana                    2022–2024                  5   
         Plana con días como regresores     2022–2024                  5   
         Tendencia plana + sábado           2022–2024                  5   
         Estacionalidad anual suave         2022–2024                  5   
         Inicial                            2022–2024                  5   
         Sábado en temporada alta           2022–2024                  5   
         Tendencia regularizada             2022–2024                  5   
Baseline Media anual                        2022–2024                  5   
         Mediana anual                      2022–2024                  5   
LightGBM Inicial                            2023–2024                  5   
                                            2024                       5   
Prophet  Plana sin patrón semanal           2022–2024                  5   

                                                           MAE (puntos de índice)  \
Familia  Configuración                      Entrenamiento                           
Prophet  Plana con días como regresores     2024                            54.91   
SARIMAX  Inicial                            2024                            57.07   
Prophet  Plana con días como regresores     2023–2024                       58.24   
SARIMAX  Inicial                            2023–2024                       58.53   
Baseline Mediana día semanal                2022–2024                       62.27   
         Naïve anual                        2022–2024                       62.06   
LightGBM Más detalle                        2022–2024                       65.15   
         Error absoluto                     2022–2024                       65.25   
Baseline Mediana mes y día                  2022–2024                       64.78   
LightGBM Calendario ampliado                2022–2024                       65.21   
         Más simple                         2022–2024                       65.00   
Prophet  Tendencia plana + anual suave      2022–2024                       66.69   
SARIMAX  SAR semanal                        2022–2024                       68.11   
         AR + SAR semanal                   2022–2024                       68.13   
         AR + SMA semanal                   2022–2024                       68.28   
         Inicial                            2022–2024                       68.79   
LightGBM Inicial                            2022–2024                       69.83   
SARIMAX  In

## Lunes–viernes, sábados y domingos

Además de los siete días por separado, agrupamos los días laborables para comparar una posible estrategia por tres grupos. El resumen conjunto calcula de nuevo los errores sobre todos sus días, sin promediar los MAPE o WAPE de cada día semanal. No seleccionamos ni registramos automáticamente un modelo combinado.

In [4]:
filas_grupos = []
for clave, grupo in predicciones.groupby(["grupo_dia"] + identidad, sort=False):
    periodo, familia, configuracion, entrenamiento = clave
    filas_grupos.append({"Grupo": periodo, "Familia": familia, "Configuración": configuracion,
                         "Entrenamiento": entrenamiento, **metricas_grupo(grupo).to_dict()})
tabla_grupos = pd.DataFrame(filas_grupos)
tabla_grupos["Días abiertos"] = tabla_grupos["Días abiertos"].astype(int)
for periodo in ["Lunes–viernes", "Sábado", "Domingo"]:
    print(f"\n{periodo}:")
    tabla = tabla_grupos.loc[tabla_grupos["Grupo"].eq(periodo)].sort_values("MAPE (%)", kind="stable")
    with pd.option_context("display.max_rows", None):
        display(tabla.drop(columns="Grupo").set_index(["Familia", "Configuración", "Entrenamiento"]).round(2))


Lunes–viernes:


Días abiertos  \
Familia  Configuración                      Entrenamiento                  
LightGBM Calendario ampliado                2022–2024                252   
         Error absoluto                     2022–2024                252   
Baseline Mediana día semanal                2022–2024                252   
LightGBM Más simple                         2022–2024                252   
         Más detalle                        2022–2024                252   
Baseline Mediana mes y día                  2022–2024                252   
SARIMAX  SAR semanal                        2022–2024                252   
         Inicial                            2022–2024                252   
         AR + SMA semanal                   2022–2024                252   
         Inicial + sábado en temporada alta 2022–2024                252   
         AR + SAR semanal                   2022–2024                252   
Prophet  Tendencia plana + sábado           2022–2024                252   
         Plana con días como regresores     2022–2024                252   
         Tendencia plana                    2022–2024                252   
         Tendencia plana + anual suave      2022–2024                252   
LightGBM Inicial                            2022–2024                252   
Prophet  Sábado en temporada alta           2022–2024                252   
         Inicial                            2022–2024                252   
SARIMAX  Inicial                            2023–2024                252   
LightGBM Inicial                            2023–2024                252   
Prophet  Plana con días como regresores     2023–2024                252   
         Estacionalidad anual suave         2022–2024                252   
         Plana con días como regresores     2024                     252   
LightGBM Inicial                            2024                     252   
SARIMAX  Inicial                            2024                     252   
Baseline Mediana anual                      2022–2024                252   
         Media anual                        2022–2024                252   
Prophet  Tendencia regularizada             2022–2024                252   
         Plana sin patrón semanal           2022–2024                252   
Baseline Naïve anual                        2022–2024                252   

                                                           MAE (puntos de índice)  \
Familia  Configuración                      Entrenamiento                           
LightGBM Calendario ampliado                2022–2024                       26.26   
         Error absoluto                     2022–2024                       26.18   
Baseline Mediana día semanal                2022–2024                       26.12   
LightGBM Más simple                         2022–2024                       25.94   
         Más detalle                        2022–2024                       26.53   
Baseline Mediana mes y día                  2022–2024                       27.03   
SARIMAX  SAR semanal                        2022–2024                       26.99   
         Inicial                            2022–2024                       27.03   
         AR + SMA semanal                   2022–2024                       27.01   
         Inicial + sábado en temporada alta 2022–2024                       26.99   
         AR + SAR semanal                   2022–2024                       27.01   
Prophet  Tendencia plana + sábado           2022–2024                       26.99   
         Plana con días como regresores     2022–2024                       27.03   
         Tendencia plana                    2022–2024                       27.03   
         Tendencia plana + anual suave      2022–2024                       26.98   
LightGBM Inicial                            2022–2024                       27.27   
Prophet  Sábado en temporada alta           2022–2024                       27.38   
         In


Sábado:


Días abiertos  \
Familia  Configuración                      Entrenamiento                  
Baseline Mediana día semanal                2022–2024                 51   
LightGBM Más simple                         2022–2024                 51   
         Error absoluto                     2022–2024                 51   
         Inicial                            2022–2024                 51   
Baseline Mediana mes y día                  2022–2024                 51   
LightGBM Más detalle                        2022–2024                 51   
Prophet  Tendencia plana + anual suave      2022–2024                 51   
         Plana con días como regresores     2022–2024                 51   
         Tendencia plana                    2022–2024                 51   
         Inicial                            2022–2024                 51   
LightGBM Inicial                            2024                      51   
SARIMAX  SAR semanal                        2022–2024                 51   
         AR + SAR semanal                   2022–2024                 51   
         Inicial                            2022–2024                 51   
LightGBM Calendario ampliado                2022–2024                 51   
SARIMAX  AR + SMA semanal                   2022–2024                 51   
Prophet  Estacionalidad anual suave         2022–2024                 51   
         Sábado en temporada alta           2022–2024                 51   
         Tendencia plana + sábado           2022–2024                 51   
LightGBM Inicial                            2023–2024                 51   
SARIMAX  Inicial + sábado en temporada alta 2022–2024                 51   
Baseline Media anual                        2022–2024                 51   
Prophet  Tendencia regularizada             2022–2024                 51   
Baseline Mediana anual                      2022–2024                 51   
SARIMAX  Inicial                            2024                      51   
Prophet  Plana con días como regresores     2024                      51   
                                            2023–2024                 51   
SARIMAX  Inicial                            2023–2024                 51   
Baseline Naïve anual                        2022–2024                 51   
Prophet  Plana sin patrón semanal           2022–2024                 51   

                                                           MAE (puntos de índice)  \
Familia  Configuración                      Entrenamiento                           
Baseline Mediana día semanal                2022–2024                       29.84   
LightGBM Más simple                         2022–2024                       30.98   
         Error absoluto                     2022–2024                       30.62   
         Inicial                            2022–2024                       30.78   
Baseline Mediana mes y día                  2022–2024                       31.84   
LightGBM Más detalle                        2022–2024                       31.91   
Prophet  Tendencia plana + anual suave      2022–2024                       31.59   
         Plana con días como regresores     2022–2024                       31.60   
         Tendencia plana                    2022–2024                       31.61   
         Inicial                            2022–2024                       31.38   
LightGBM Inicial                            2024                            30.78   
SARIMAX  SAR semanal                        2022–2024                       32.01   
         AR + SAR semanal                   2022–2024                       32.02   
         Inicial                            2022–2024                       32.07   
LightGBM Calendario ampliado                2022–2024                       33.04   
SARIMAX  AR + SMA semanal                   2022–2024                       32.06   
Prophet  Estacionalidad anual suave         2022–2024                       31.69   
         Sá


Domingo:


Días abiertos  \
Familia  Configuración                      Entrenamiento                  
Prophet  Plana con días como regresores     2024                       5   
SARIMAX  Inicial                            2024                       5   
Prophet  Plana con días como regresores     2023–2024                  5   
SARIMAX  Inicial                            2023–2024                  5   
Baseline Mediana día semanal                2022–2024                  5   
         Naïve anual                        2022–2024                  5   
LightGBM Más detalle                        2022–2024                  5   
         Error absoluto                     2022–2024                  5   
Baseline Mediana mes y día                  2022–2024                  5   
LightGBM Calendario ampliado                2022–2024                  5   
         Más simple                         2022–2024                  5   
Prophet  Tendencia plana + anual suave      2022–2024                  5   
SARIMAX  SAR semanal                        2022–2024                  5   
         AR + SAR semanal                   2022–2024                  5   
         AR + SMA semanal                   2022–2024                  5   
         Inicial                            2022–2024                  5   
LightGBM Inicial                            2022–2024                  5   
SARIMAX  Inicial + sábado en temporada alta 2022–2024                  5   
Prophet  Tendencia plana                    2022–2024                  5   
         Plana con días como regresores     2022–2024                  5   
         Tendencia plana + sábado           2022–2024                  5   
         Estacionalidad anual suave         2022–2024                  5   
         Inicial                            2022–2024                  5   
         Sábado en temporada alta           2022–2024                  5   
         Tendencia regularizada             2022–2024                  5   
Baseline Media anual                        2022–2024                  5   
         Mediana anual                      2022–2024                  5   
LightGBM Inicial                            2023–2024                  5   
                                            2024                       5   
Prophet  Plana sin patrón semanal           2022–2024                  5   

                                                           MAE (puntos de índice)  \
Familia  Configuración                      Entrenamiento                           
Prophet  Plana con días como regresores     2024                            54.91   
SARIMAX  Inicial                            2024                            57.07   
Prophet  Plana con días como regresores     2023–2024                       58.24   
SARIMAX  Inicial                            2023–2024                       58.53   
Baseline Mediana día semanal                2022–2024                       62.27   
         Naïve anual                        2022–2024                       62.06   
LightGBM Más detalle                        2022–2024                       65.15   
         Error absoluto                     2022–2024                       65.25   
Baseline Mediana mes y día                  2022–2024                       64.78   
LightGBM Calendario ampliado                2022–2024                       65.21   
         Más simple                         2022–2024                       65.00   
Prophet  Tendencia plana + anual suave      2022–2024                       66.69   
SARIMAX  SAR semanal                        2022–2024                       68.11   
         AR + SAR semanal                   2022–2024                       68.13   
         AR + SMA semanal                   2022–2024                       68.28   
         Inicial                            2022–2024                       68.79   
LightGBM Inicial                            2022–2024                       69.83   
SARIMAX  In

## Comparación por mes

Agrupamos las mismas predicciones y los mismos días abiertos de 2025 por mes. Las métricas se calculan directamente sobre los días abiertos de cada mes; no son promedios de las métricas por día de la semana. Primero se compara la mediana semanal con la configuración de menor MAPE y después se muestran todas las configuraciones, con MAPE, WAPE, MAE y sesgo.

La selección del menor error en cada mes es retrospectiva: no demuestra que esa configuración vaya a ser la mejor en el futuro. Los meses contienen menos observaciones que el año completo.

In [7]:
MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
filas_meses = []
for clave, grupo in predicciones.groupby([predicciones["fecha"].dt.month] + identidad, sort=False):
    numero, familia, configuracion, entrenamiento = clave
    filas_meses.append({"Número de mes": numero, "Mes": MESES[numero - 1], "Familia": familia,
                        "Configuración": configuracion, "Entrenamiento": entrenamiento,
                        **metricas_grupo(grupo).to_dict()})
tabla_meses = pd.DataFrame(filas_meses)
tabla_meses["Días abiertos"] = tabla_meses["Días abiertos"].astype(int)

display(resumir_mejores(tabla_meses, "Mes", MESES).round(2))

,Días abiertos,Baseline MAPE (%),Baseline WAPE (%),Mejor familia por MAPE,Mejor configuración por MAPE,Entrenamiento,Mejor MAPE (%),WAPE de esa configuración (%)
Mes,,,,,,,,
Enero,25,33.89,26.15,Baseline,Mediana mes y día,2022–2024,21.54,20.35
Febrero,24,24.85,24.82,LightGBM,Error absoluto,2022–2024,21.53,23.18
Marzo,27,37.46,27.75,LightGBM,Calendario ampliado,2022–2024,30.75,25.51
Abril,28,45.05,31.52,SARIMAX,Inicial,2024,41.06,28.96
Mayo,26,34.27,29.32,LightGBM,Calendario ampliado,2022–2024,32.64,27.94
Junio,24,29.22,25.54,LightGBM,Más detalle,2022–2024,23.22,22.03
Julio,27,34.81,28.46,LightGBM,Inicial,2024,30.57,26.79
Agosto,24,18.36,19.34,Baseline,Mediana día semanal,2022–2024,18.36,19.34
Septiembre,26,26.43,26.05,Baseline,Mediana día semanal,2022–2024,26.43,26.05


In [5]:
for numero, mes in enumerate(MESES, start=1):
    print(f"\n{mes}:")
    tabla = tabla_meses.loc[tabla_meses["Número de mes"].eq(numero)].sort_values("MAPE (%)", kind="stable")
    with pd.option_context("display.max_rows", None):
        display(tabla.drop(columns=["Número de mes", "Mes"]).set_index(["Familia", "Configuración", "Entrenamiento"]).round(2))


Enero:


Días abiertos  \
Familia  Configuración                      Entrenamiento                  
Baseline Mediana mes y día                  2022–2024                 25   
LightGBM Más detalle                        2022–2024                 25   
         Error absoluto                     2022–2024                 25   
         Más simple                         2022–2024                 25   
Baseline Mediana anual                      2022–2024                 25   
LightGBM Calendario ampliado                2022–2024                 25   
Baseline Media anual                        2022–2024                 25   
LightGBM Inicial                            2022–2024                 25   
                                            2023–2024                 25   
Prophet  Plana con días como regresores     2022–2024                 25   
         Tendencia plana                    2022–2024                 25   
LightGBM Inicial                            2024                      25   
SARIMAX  AR + SMA semanal                   2022–2024                 25   
Prophet  Tendencia plana + sábado           2022–2024                 25   
SARIMAX  SAR semanal                        2022–2024                 25   
         Inicial                            2022–2024                 25   
         Inicial + sábado en temporada alta 2022–2024                 25   
         Inicial                            2024                      25   
         AR + SAR semanal                   2022–2024                 25   
         Inicial                            2023–2024                 25   
Prophet  Plana con días como regresores     2023–2024                 25   
                                            2024                      25   
         Inicial                            2022–2024                 25   
         Sábado en temporada alta           2022–2024                 25   
         Tendencia regularizada             2022–2024                 25   
Baseline Naïve anual                        2022–2024                 25   
Prophet  Tendencia plana + anual suave      2022–2024                 25   
         Estacionalidad anual suave         2022–2024                 25   
         Plana sin patrón semanal           2022–2024                 25   
Baseline Mediana día semanal                2022–2024                 25   

                                                           MAE (puntos de índice)  \
Familia  Configuración                      Entrenamiento                           
Baseline Mediana mes y día                  2022–2024                       17.83   
LightGBM Más detalle                        2022–2024                       17.88   
         Error absoluto                     2022–2024                       17.96   
         Más simple                         2022–2024                       18.03   
Baseline Mediana anual                      2022–2024                       17.62   
LightGBM Calendario ampliado                2022–2024                       18.83   
Baseline Media anual                        2022–2024                       18.96   
LightGBM Inicial                            2022–2024                       18.31   
                                            2023–2024                       18.48   
Prophet  Plana con días como regresores     2022–2024                       18.31   
         Tendencia plana                    2022–2024                       18.31   
LightGBM Inicial                            2024                            18.67   
SARIMAX  AR + SMA semanal                   2022–2024                       18.53   
Prophet  Tendencia plana + sábado           2022–2024                       18.33   
SARIMAX  SAR semanal                        2022–2024                       18.53   
         Inicial                            2022–2024                       18.63   
         Inicial + sábado en temporada alta 2022–2024                       18.71   
         In


Febrero:


Días abiertos  \
Familia  Configuración                      Entrenamiento                  
LightGBM Error absoluto                     2022–2024                 24   
Prophet  Tendencia plana + anual suave      2022–2024                 24   
LightGBM Inicial                            2022–2024                 24   
         Más detalle                        2022–2024                 24   
Prophet  Plana con días como regresores     2022–2024                 24   
         Tendencia plana                    2022–2024                 24   
         Estacionalidad anual suave         2022–2024                 24   
Baseline Mediana mes y día                  2022–2024                 24   
SARIMAX  Inicial                            2022–2024                 24   
LightGBM Más simple                         2022–2024                 24   
SARIMAX  AR + SMA semanal                   2022–2024                 24   
         SAR semanal                        2022–2024                 24   
         AR + SAR semanal                   2022–2024                 24   
Prophet  Tendencia plana + sábado           2022–2024                 24   
SARIMAX  Inicial + sábado en temporada alta 2022–2024                 24   
Baseline Media anual                        2022–2024                 24   
LightGBM Inicial                            2023–2024                 24   
         Calendario ampliado                2022–2024                 24   
Prophet  Inicial                            2022–2024                 24   
Baseline Mediana día semanal                2022–2024                 24   
Prophet  Sábado en temporada alta           2022–2024                 24   
Baseline Mediana anual                      2022–2024                 24   
Prophet  Tendencia regularizada             2022–2024                 24   
LightGBM Inicial                            2024                      24   
Prophet  Plana con días como regresores     2023–2024                 24   
SARIMAX  Inicial                            2023–2024                 24   
                                            2024                      24   
Prophet  Plana con días como regresores     2024                      24   
         Plana sin patrón semanal           2022–2024                 24   
Baseline Naïve anual                        2022–2024                 24   

                                                           MAE (puntos de índice)  \
Familia  Configuración                      Entrenamiento                           
LightGBM Error absoluto                     2022–2024                       24.82   
Prophet  Tendencia plana + anual suave      2022–2024                       27.29   
LightGBM Inicial                            2022–2024                       24.90   
         Más detalle                        2022–2024                       26.36   
Prophet  Plana con días como regresores     2022–2024                       26.07   
         Tendencia plana                    2022–2024                       26.07   
         Estacionalidad anual suave         2022–2024                       27.07   
Baseline Mediana mes y día                  2022–2024                       26.69   
SARIMAX  Inicial                            2022–2024                       26.21   
LightGBM Más simple                         2022–2024                       26.61   
SARIMAX  AR + SMA semanal                   2022–2024                       26.19   
         SAR semanal                        2022–2024                       26.22   
         AR + SAR semanal                   2022–2024                       26.28   
Prophet  Tendencia plana + sábado           2022–2024                       26.64   
SARIMAX  Inicial + sábado en temporada alta 2022–2024                       26.79   
Baseline Media anual                        2022–2024                       27.59   
LightGBM Inicial                            2023–2024                       25.82   
         Ca


Marzo:


Días abiertos  \
Familia  Configuración                      Entrenamiento                  
LightGBM Calendario ampliado                2022–2024                 27   
         Error absoluto                     2022–2024                 27   
         Más detalle                        2022–2024                 27   
Baseline Mediana mes y día                  2022–2024                 27   
LightGBM Inicial                            2022–2024                 27   
         Más simple                         2022–2024                 27   
Prophet  Tendencia plana                    2022–2024                 27   
         Plana con días como regresores     2022–2024                 27   
SARIMAX  AR + SAR semanal                   2022–2024                 27   
         SAR semanal                        2022–2024                 27   
         AR + SMA semanal                   2022–2024                 27   
         Inicial                            2022–2024                 27   
Prophet  Tendencia plana + sábado           2022–2024                 27   
SARIMAX  Inicial + sábado en temporada alta 2022–2024                 27   
Baseline Mediana anual                      2022–2024                 27   
Prophet  Inicial                            2022–2024                 27   
         Sábado en temporada alta           2022–2024                 27   
Baseline Mediana día semanal                2022–2024                 27   
SARIMAX  Inicial                            2023–2024                 27   
Prophet  Tendencia regularizada             2022–2024                 27   
         Plana con días como regresores     2023–2024                 27   
         Tendencia plana + anual suave      2022–2024                 27   
LightGBM Inicial                            2023–2024                 27   
Baseline Media anual                        2022–2024                 27   
Prophet  Estacionalidad anual suave         2022–2024                 27   
         Plana sin patrón semanal           2022–2024                 27   
         Plana con días como regresores     2024                      27   
SARIMAX  Inicial                            2024                      27   
LightGBM Inicial                            2024                      27   
Baseline Naïve anual                        2022–2024                 27   

                                                           MAE (puntos de índice)  \
Familia  Configuración                      Entrenamiento                           
LightGBM Calendario ampliado                2022–2024                       25.88   
         Error absoluto                     2022–2024                       25.56   
         Más detalle                        2022–2024                       25.19   
Baseline Mediana mes y día                  2022–2024                       25.85   
LightGBM Inicial                            2022–2024                       26.19   
         Más simple                         2022–2024                       27.60   
Prophet  Tendencia plana                    2022–2024                       26.81   
         Plana con días como regresores     2022–2024                       26.81   
SARIMAX  AR + SAR semanal                   2022–2024                       26.71   
         SAR semanal                        2022–2024                       26.70   
         AR + SMA semanal                   2022–2024                       26.73   
         Inicial                            2022–2024                       26.71   
Prophet  Tendencia plana + sábado           2022–2024                       27.24   
SARIMAX  Inicial + sábado en temporada alta 2022–2024                       27.05   
Baseline Mediana anual                      2022–2024                       28.98   
Prophet  Inicial                            2022–2024                       27.43   
         Sábado en temporada alta           2022–2024                       27.83   
Baseline Me

... (14 outputs left out)


Noviembre:


Días abiertos  \
Familia  Configuración                      Entrenamiento                  
SARIMAX  Inicial                            2024                      24   
Prophet  Plana con días como regresores     2024                      24   
LightGBM Inicial                            2024                      24   
Baseline Mediana anual                      2022–2024                 24   
Prophet  Plana con días como regresores     2023–2024                 24   
SARIMAX  Inicial                            2023–2024                 24   
LightGBM Error absoluto                     2022–2024                 24   
Prophet  Plana con días como regresores     2022–2024                 24   
         Tendencia plana                    2022–2024                 24   
         Inicial                            2022–2024                 24   
         Sábado en temporada alta           2022–2024                 24   
         Tendencia plana + sábado           2022–2024                 24   
Baseline Mediana mes y día                  2022–2024                 24   
LightGBM Calendario ampliado                2022–2024                 24   
         Más detalle                        2022–2024                 24   
         Inicial                            2023–2024                 24   
         Más simple                         2022–2024                 24   
Baseline Media anual                        2022–2024                 24   
         Mediana día semanal                2022–2024                 24   
SARIMAX  Inicial                            2022–2024                 24   
         AR + SAR semanal                   2022–2024                 24   
         AR + SMA semanal                   2022–2024                 24   
         SAR semanal                        2022–2024                 24   
         Inicial + sábado en temporada alta 2022–2024                 24   
LightGBM Inicial                            2022–2024                 24   
Baseline Naïve anual                        2022–2024                 24   
Prophet  Tendencia regularizada             2022–2024                 24   
         Tendencia plana + anual suave      2022–2024                 24   
         Estacionalidad anual suave         2022–2024                 24   
         Plana sin patrón semanal           2022–2024                 24   

                                                           MAE (puntos de índice)  \
Familia  Configuración                      Entrenamiento                           
SARIMAX  Inicial                            2024                            19.31   
Prophet  Plana con días como regresores     2024                            20.37   
LightGBM Inicial                            2024                            20.23   
Baseline Mediana anual                      2022–2024                       20.38   
Prophet  Plana con días como regresores     2023–2024                       20.04   
SARIMAX  Inicial                            2023–2024                       21.30   
LightGBM Error absoluto                     2022–2024                       22.00   
Prophet  Plana con días como regresores     2022–2024                       21.85   
         Tendencia plana                    2022–2024                       21.85   
         Inicial                            2022–2024                       21.85   
         Sábado en temporada alta           2022–2024                       21.99   
         Tendencia plana + sábado           2022–2024                       22.02   
Baseline Mediana mes y día                  2022–2024                       23.10   
LightGBM Calendario ampliado                2022–2024                       22.23   
         Más detalle                        2022–2024                       23.12   
         Inicial                            2023–2024                       22.98   
         Más simple                         2022–2024                       22.58   
Baseline Me


Diciembre:


Días abiertos  \
Familia  Configuración                      Entrenamiento                  
Baseline Mediana día semanal                2022–2024                 26   
         Mediana mes y día                  2022–2024                 26   
LightGBM Más simple                         2022–2024                 26   
Prophet  Plana con días como regresores     2024                      26   
LightGBM Inicial                            2024                      26   
         Más detalle                        2022–2024                 26   
Prophet  Tendencia plana + anual suave      2022–2024                 26   
LightGBM Calendario ampliado                2022–2024                 26   
SARIMAX  Inicial                            2024                      26   
Prophet  Estacionalidad anual suave         2022–2024                 26   
LightGBM Error absoluto                     2022–2024                 26   
Baseline Naïve anual                        2022–2024                 26   
         Mediana anual                      2022–2024                 26   
         Media anual                        2022–2024                 26   
Prophet  Plana con días como regresores     2023–2024                 26   
LightGBM Inicial                            2022–2024                 26   
Prophet  Sábado en temporada alta           2022–2024                 26   
LightGBM Inicial                            2023–2024                 26   
Prophet  Tendencia plana + sábado           2022–2024                 26   
SARIMAX  Inicial                            2023–2024                 26   
Prophet  Inicial                            2022–2024                 26   
         Plana con días como regresores     2022–2024                 26   
         Tendencia plana                    2022–2024                 26   
SARIMAX  Inicial + sábado en temporada alta 2022–2024                 26   
         SAR semanal                        2022–2024                 26   
         AR + SAR semanal                   2022–2024                 26   
         AR + SMA semanal                   2022–2024                 26   
         Inicial                            2022–2024                 26   
Prophet  Tendencia regularizada             2022–2024                 26   
         Plana sin patrón semanal           2022–2024                 26   

                                                           MAE (puntos de índice)  \
Familia  Configuración                      Entrenamiento                           
Baseline Mediana día semanal                2022–2024                       26.10   
         Mediana mes y día                  2022–2024                       31.85   
LightGBM Más simple                         2022–2024                       32.85   
Prophet  Plana con días como regresores     2024                            35.48   
LightGBM Inicial                            2024                            32.80   
         Más detalle                        2022–2024                       34.89   
Prophet  Tendencia plana + anual suave      2022–2024                       35.17   
LightGBM Calendario ampliado                2022–2024                       35.34   
SARIMAX  Inicial                            2024                            36.27   
Prophet  Estacionalidad anual suave         2022–2024                       36.30   
LightGBM Error absoluto                     2022–2024                       37.80   
Baseline Naïve anual                        2022–2024                       43.30   
         Mediana anual                      2022–2024                       40.19   
         Media anual                        2022–2024                       40.49   
Prophet  Plana con días como regresores     2023–2024                       40.12   
LightGBM Inicial                            2022–2024                       38.73   
Prophet  Sábado en temporada alta           2022–2024                       39.81   
LightGBM In